# S08 — one of six independent dense workers
Open this notebook in six CUDA sessions with WORKER_ID 0–5. Every session uses the matching worker ZIP from one shared immutable plan. Each worker has its own local and Drive output. Never launch two writers for the same worker. No distributed training/NCCL is used. See docs/RETRIEVAL_STAGES7_8.md. All tests/jobs are owner-run; QA/calibration/comparisons remain deferred.

Cell 4 streams stdout/stderr and saves a timestamped log locally and under DRIVE_RUN/logs/worker_ID. For an encode failure, share the original error above the final RuntimeError or the last 80 log lines. The updated notebook can be opened alongside existing worker ZIPs; no encoding source, plan or completed batch identity is changed.

Fast-tokenizer compatibility update: setup installs an explicit encoder adapter embedded in this notebook; use the same update for all six workers and query encoding. No ZIP regeneration is required. The five original plan-bound modules remain unchanged and all original checksum checks still run. The adapter prepares the special-token mask; model input IDs, mapped windows, pooling, source IDs and checkpoints retain their original policy. Its version and source checksum are recorded in newly encoded model/vector registries.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
WORKER_ID = 0  # unique in each session: 0, 1, 2, 3, 4, 5
DRIVE_RUN = Path('/content/drive/MyDrive/ViBioMIR/dense_stage8_bge_structure')
BUNDLE_DIR = DRIVE_RUN / 'bundles'
BUNDLE = BUNDLE_DIR / f'worker_{WORKER_ID}.zip'
PROJECT = Path('/content') / f'vibiomir_dense_worker_{WORKER_ID}'
RUN_DIR = Path('/content') / f'dense_worker_{WORKER_ID}'
BACKUP_DIR = DRIVE_RUN / f'worker_{WORKER_ID}'
BATCH_SIZE = 16  # T4 start; L4:32, A100:64 after pilot
PADDED_TOKENS = 8192  # L4:16384, A100:32768 after pilot
WAVE_CHUNKS = 128
STOP_AFTER_BATCHES = 1  # then None; rerun the encode cell
assert BUNDLE.is_file(), 'Upload this worker ZIP and edit DRIVE_RUN first.'
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')


In [ ]:
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    meta = json.loads(archive.read('BUNDLE.json'))
    assert meta['worker_id'] == WORKER_ID, 'Wrong worker ZIP'
    required = sum(info.file_size for info in archive.infolist())
    assert shutil.disk_usage('/content').free > required * 3 + 8 * 1024**3, 'Need local space for model, input and worker output.'
    for info in archive.infolist():
        assert PROJECT.resolve() in (PROJECT / info.filename).resolve().parents, 'Unsafe ZIP path'
        assert (info.external_attr >> 16) & 0o170000 != 0o120000, 'Symlink ZIP entry'
    archive.extractall(PROJECT)
# Compatibility overlay for existing frozen ZIPs. Plan-bound files are unchanged.
DENSE_COMPAT_SOURCE = '"""Explicit fast-tokenizer compatibility runner for existing frozen S08 plans.\n\nThe five plan-bound source files stay unchanged. The runtime adapter and its\nchecksum are recorded in each newly created encoder registry/vector batch.\n"""\nfrom pathlib import Path\n\nfrom .dense_models import DenseEncoder\nfrom .io import file_hash\nfrom .retrieval_contracts import require\n\n\nCOMPATIBILITY_ID = \'s08-fast-special-mask-v1\'\nENCODER_SHA256 = \'f0cd5f5466433639830945a7fbe989d4fad3c43a9f7ce54823109a873bdf9b70\'\n\n\nclass SpecialMaskTokenizer:\n    """Delegate tokenization; mark only special positions inserted by the model.\n\n    Fast tokenizers cannot prepare_for_model(return_special_tokens_mask=True).\n    A negative sentinel through the same input template locates inserted tokens\n    without classifying unknown/literal special IDs in the source as padding.\n    """\n\n    def __init__(self, tokenizer):\n        self.tokenizer = tokenizer\n\n    def __getattr__(self, name):\n        return getattr(self.tokenizer, name)\n\n    def __call__(self, *args, **kwargs):\n        return self.tokenizer(*args, **kwargs)\n\n    def prepare_for_model(self, ids, pair_ids=None, **kwargs):\n        requested = kwargs.pop(\'return_special_tokens_mask\', False)\n        feature = self.tokenizer.prepare_for_model(\n            ids, pair_ids=pair_ids, return_special_tokens_mask=False, **kwargs)\n        if requested:\n            require(pair_ids is None and all(i >= 0 for i in ids), \'Compatibility adapter requires one nonnegative token sequence\')\n            template = self.tokenizer.build_inputs_with_special_tokens([-1] * len(ids)) if kwargs.get(\'add_special_tokens\', True) else [-1] * len(ids)\n            mask = [int(i != -1) for i in template]\n            require(len(mask) == len(feature[\'input_ids\']) and\n                    [i for i, special in zip(feature[\'input_ids\'], mask) if not special] == list(ids),\n                    \'Special-token template changed token IDs or truncated input\')\n            feature[\'special_tokens_mask\'] = mask\n        return feature\n\n\ndef compatibility_identity():\n    return {\'id\': COMPATIBILITY_ID, \'sha256\': file_hash(__file__),\n            \'encoder_source_sha256\': ENCODER_SHA256,\n            \'policy\': \'unchanged token IDs/windows; mask inserted template positions only\'}\n\n\nclass CompatibleDenseEncoder(DenseEncoder):\n    def __init__(self, *args, **kwargs):\n        require(file_hash(Path(__file__).with_name(\'dense_models.py\')) == ENCODER_SHA256,\n                \'This compatibility adapter targets the original S08 encoder; review it before using another revision\')\n        super().__init__(*args, **kwargs)\n        self.tokenizer = SpecialMaskTokenizer(self.tokenizer)\n        self.registry[\'tokenizer_compatibility\'] = compatibility_identity()\n\n\ndef main():\n    from . import dense_stage8\n    # read_plan still checks the original five source hashes. Only encoder\n    # construction changes; source, model, pooling and checkpoints stay bound.\n    dense_stage8.DenseEncoder = CompatibleDenseEncoder\n    dense_stage8.main()\n\n\nif __name__ == \'__main__\':\n    main()\n'
(PROJECT / 'vibiomir/dense_tokenizer_compat.py').write_text(DENSE_COMPAT_SOURCE, encoding='utf-8')
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-stage8-colab.txt'], check=True)
print('If a runtime restart is requested, restart and rerun setup.')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime; TPU is not supported by this worker.'
plan = json.loads((PROJECT / 'plan.json').read_text())
assert plan['signature'] == meta['plan_signature']
print({'gpu': torch.cuda.get_device_name(0), 'worker': plan['workers'][WORKER_ID], 'encoding':plan['encoding']})
# Check the actual bundled fast tokenizer before loading model weights.
from transformers import AutoTokenizer
from vibiomir.dense_models import token_windows
from vibiomir.dense_tokenizer_compat import SpecialMaskTokenizer, compatibility_identity
fast_tokenizer = AutoTokenizer.from_pretrained(PROJECT / 'tokenizer', local_files_only=True, use_fast=True)
for example in ('Không dùng 5 mg khi sốt. ', 'No fever; dose 5 mg. ', '肾结石治疗。'):
    text = example * 20
    ids = fast_tokenizer(text, add_special_tokens=False)['input_ids']
    windows = token_windows(SpecialMaskTokenizer(fast_tokenizer), text, 32, 4)
    covered = set()
    for window in windows:
        features = window['features']
        content = [i for i, mask in zip(features['input_ids'], features['special_tokens_mask']) if not mask]
        assert content == ids[window['token_start']:window['token_end']], 'Tokenizer adapter changed input IDs'
        covered.update(range(window['token_start'], window['token_end']))
    assert covered == set(range(len(ids))), 'Incomplete tokenizer window coverage'
print('Real fast-tokenizer preflight passed:', compatibility_identity())
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_stage78*.py', '-v'], check=True)


In [ ]:
# Stream the child process output into Colab and keep its original error.
from collections import deque
from datetime import datetime, timezone

command = [sys.executable, '-u', '-m', 'vibiomir.dense_tokenizer_compat', 'encode-worker',
    '--plan', str(PROJECT / 'plan.json'), '--chunk-dir', str(PROJECT / 'chunks'),
    '--tokenizer-dir', str(PROJECT / 'tokenizer'), '--output-dir', str(RUN_DIR),
    '--worker-id', str(WORKER_ID), '--backup-dir', str(BACKUP_DIR),
    '--batch-size', str(BATCH_SIZE), '--padded-tokens', str(PADDED_TOKENS), '--wave-chunks', str(WAVE_CHUNKS)]
if STOP_AFTER_BATCHES is not None:
    command += ['--stop-after-batches', str(STOP_AFTER_BATCHES)]
# Logs are outside RUN_DIR: creating files there before the first run would
# violate the worker's requirement for a new empty output directory.
LOG_PATH = PROJECT / f"encode_worker_{WORKER_ID}_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}.log"
error_tail = deque(maxlen=80)
print('Worker log:', LOG_PATH, flush=True)
with LOG_PATH.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            error_tail.append(line)
        returncode = process.wait()
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()
try:
    saved_logs = DRIVE_RUN / 'logs' / f'worker_{WORKER_ID}'
    saved_logs.mkdir(parents=True, exist_ok=True)
    shutil.copy2(LOG_PATH, saved_logs / LOG_PATH.name)
    print('Log copied to:', saved_logs / LOG_PATH.name, flush=True)
except OSError as exc:
    print('Could not copy log to Drive; local log remains at', LOG_PATH, ':', exc, flush=True)
if returncode:
    print('\nLast worker output:\n' + ''.join(error_tail), flush=True)
    raise RuntimeError(f'Worker {WORKER_ID} exited with code {returncode}. '
                       f'The original error is printed above and saved in {LOG_PATH}. '
                       'Keep the existing plan, bundle and committed batch folders.')
state = json.loads((RUN_DIR / 'manifest.json').read_text())
print('Worker state:', state['state'])
if state['state'] != 'complete':
    print('Set STOP_AFTER_BATCHES=None and rerun this cell. Committed batches are reused; Drive restore is automatic after reset.')


In [ ]:
state = json.loads((RUN_DIR / 'manifest.json').read_text())
if state['state'] == 'complete':
    subprocess.run([sys.executable, '-m', 'vibiomir.dense_stage8', 'validate-worker',
        '--plan', str(PROJECT / 'plan.json'), '--chunk-dir', str(PROJECT / 'chunks'),
        '--output-dir', str(RUN_DIR), '--worker-id', str(WORKER_ID)], check=True)
    shutil.copy2(RUN_DIR / 'VALIDATION.json', BACKUP_DIR / 'VALIDATION.json')
    print('Complete worker persisted:', BACKUP_DIR)
else:
    print('Finish all assigned batches before validation.')


In [ ]:
ENCODE_QUERIES = WORKER_ID == 0
query_files = [p for p in (PROJECT / 'queries.parquet', PROJECT / 'queries.json') if p.is_file()]
if ENCODE_QUERIES and query_files:
    QUERY_DIR = Path('/content/dense_query_vectors')
    subprocess.run([sys.executable, '-m', 'vibiomir.dense_tokenizer_compat', 'encode-queries',
        '--plan', str(PROJECT / 'plan.json'), '--tokenizer-dir', str(PROJECT / 'tokenizer'),
        '--queries', str(query_files[0]), '--output-dir', str(QUERY_DIR), '--batch-size', str(BATCH_SIZE)], check=True)
    shutil.copytree(QUERY_DIR, DRIVE_RUN / 'query_vectors', dirs_exist_ok=True)
    print('Query vectors copied to Drive. No relevance evaluation performed.')


## Optional central merge on one session
Run only after all six worker manifests are complete. This is CPU/disk work; the other five GPUs can stop. Local merge is also documented. Source chunks from all ZIPs are needed to validate every mapping. Merged output copies vector batches, so both worker copies and merged data need local disk. Final comparisons and ANN recall measurement are deferred.

In [ ]:
MERGE_ALL = False  # enable on one session only, once all workers finish
if MERGE_ALL:
    assert len(plan['workers']) == 6
    expected_bytes = plan['source']['chunks'] * plan['encoding']['model_spec']['dimension'] * 2
    assert shutil.disk_usage('/content').free > expected_bytes * 3 + 10 * 1024**3, 'Merge locally instead if Colab disk is insufficient.'
    for worker in plan['workers']:
        source = DRIVE_RUN / f"worker_{worker['id']}"
        assert (source / 'manifest.json').is_file(), f'Missing worker: {source}'
        manifest = json.loads((source / 'manifest.json').read_text())
        assert manifest['state'] == 'complete' and manifest['identity']['plan_signature'] == plan['signature'], 'Incomplete/mismatched worker'
    MERGE_WORKERS = Path('/content/dense_merge_workers')
    MERGED = Path('/content/dense_merged')
    for worker in plan['workers']:
        with zipfile.ZipFile(BUNDLE_DIR / f"worker_{worker['id']}.zip") as archive:
            assert json.loads(archive.read('plan.json')) == plan, 'Mixed plans'
            for info in archive.infolist():
                if info.filename.startswith('chunks/'):
                    assert PROJECT.resolve() in (PROJECT / info.filename).resolve().parents
                    archive.extract(info, PROJECT)
        print('Restoring worker', worker['id'], flush=True)
        shutil.copytree(DRIVE_RUN / f"worker_{worker['id']}", MERGE_WORKERS / f"worker_{worker['id']}", dirs_exist_ok=True)
    subprocess.run([sys.executable, '-m', 'vibiomir.dense_stage8', 'merge',
        '--plan', str(PROJECT / 'plan.json'), '--chunk-dir', str(PROJECT / 'chunks'),
        '--workers-dir', str(MERGE_WORKERS), '--output-dir', str(MERGED)], check=True)
    shutil.copytree(MERGED, DRIVE_RUN / 'merged', dirs_exist_ok=True)
    print('Merged vectors/mapping/validation persisted:', DRIVE_RUN / 'merged')
